In [1]:
import contextlib, copy, math, time
from types import SimpleNamespace

from tinygrad import Tensor, nn, dtypes
from tinygrad.helpers import GlobalCounters, colored

In [54]:
def time_to_str(t:float, w=9) -> str: return colored(next((f"{t * d:{w}.2f} {pr}" for d,pr in [(1, "s "),(1e3, "ms")] if t > 10/d), f"{t * 1e6:{w}.2f} us"), 'yellow')
def op_to_str(op:int, w=9) -> str: return colored(next((f"{op / d:{w}.2f} {pr}" for d,pr in [(1e12, "TOPs"), (1e9, "GOPs")] if op >= d), f"{op:{w}} OPs"), 'yellow')
def mem_to_str(mem:int, w=9) -> str: return colored(next((f"{mem / d:{w}.2f} {pr}" for d,pr in [(1e9, "GB"), (1e6, "MB")] if mem >= d), f"{mem:{w}} bytes"), 'yellow')
def op_tm_to_str(x:float, w=9) -> str: return colored(next((f"{x / d:{w}.2f} {pr}" for d,pr in [(1e12, "TOPs/sec"), (1e9, "GOPs/sec"), (1e6, "MOPs/sec")] if x >= d), f"{x:{w}.2f} OPs/sec"), 'blue')
def mem_tm_to_str(x:float, w=9) -> str: return colored(next((f"{x / d:{w}.2f} {pr}" for d,pr in [(1e9, "GB/sec"), (1e6, "MB/sec")] if x >= d), f"{x:{w}.2f} bytes/sec"), 'blue')
def op_mem_to_str(x:float, w=9) -> str: return colored(next((f"{x / d:{w}.2f} {pr}" for d,pr in [(1e9, "GOPs/GB"), (1e6, "MOPs/MB")] if x >= d), f"{x:{w}.2f} OPs/byte"), 'blue')

In [55]:
class Stats(contextlib.ContextDecorator):
  def __init__(self, prefix="", enabled=True): self.prefix, self.enabled = prefix, enabled
  def __enter__(self): self.tm, self.op, self.mem = time.perf_counter_ns(), GlobalCounters.global_ops, GlobalCounters.global_mem
  def __exit__(self, *exc):
    if not self.enabled: return
    tm, op, mem = (time.perf_counter_ns()-self.tm)*1e-9, GlobalCounters.global_ops-self.op, GlobalCounters.global_mem-self.mem
    op_tm, mem_tm, op_mem = op/(tm or 1e-20), mem/(tm or 1e-20), op/(mem or 1e-20)
    print(f"{self.prefix:<20}\t{time_to_str(tm)}, {op_to_str(op)}, {mem_to_str(mem)}, {op_tm_to_str(op_tm)}, {mem_tm_to_str(mem_tm)}, {op_mem_to_str(op_mem)}")

In [56]:
def apply_rope(x:Tensor, start_pos:int, base:float = 10000.0) -> Tensor:
  B, H, T, Dh = x.shape
  assert (Dh & 1) == 0, "RoPE requires an even head Dension"
  half = Dh // 2
  angles = (Tensor.arange(T, dtype="float32") + start_pos)[:, None] * (base ** (-(Tensor.arange(half, dtype="float32") / half)))[None, :]
  cos, sin = angles.cos().reshape(1, 1, T, half).cast(x.dtype), angles.sin().reshape(1, 1, T, half).cast(x.dtype)
  x_pairs = x.reshape(B, H, T, half, 2)
  return Tensor.stack(x_pairs[..., 0] * cos - x_pairs[..., 1] * sin,
                      x_pairs[..., 0] * sin + x_pairs[..., 1] * cos, dim=-1).reshape(B, H, T, Dh)

# Multi-Headed Attention (MHA)

In [57]:
# multi-headed attention
# https://arxiv.org/abs/1706.03762
class MHA:
  def __init__(self, dim:int, num_heads:int, max_context:int=0):
    self.num_heads = num_heads # H
    self.head_dim = dim // num_heads # Dh
    self.max_context = max_context

    self.attn_q = nn.Linear(dim, dim, bias=False) # (D,D)
    self.attn_k = nn.Linear(dim, dim, bias=False) # (D,D)
    self.attn_v = nn.Linear(dim, dim, bias=False) # (D,D)
    self.attn_o = nn.Linear(dim, dim, bias=False) # (D,D)

  def __call__(self, x:Tensor, start_pos:int=0, is_causal=True) -> Tensor:
    # projections
    q, k, v = self.attn_q(x), self.attn_k(x), self.attn_v(x) # (B,T,D) -> (B,T,D)

    # reshape
    B, T, _ = x.shape
    q = q.reshape(B, T, self.num_heads, self.head_dim).transpose(1, 2)  # (B,T,D) -> (B,H,T,Dh)
    k = k.reshape(B, T, self.num_heads, self.head_dim).transpose(1, 2)  # (B,T,D) -> (B,H,T,Dh)
    v = v.reshape(B, T, self.num_heads, self.head_dim).transpose(1, 2)  # (B,T,D) -> (B,H,T,Dh)

    # positional embeddings
    q, k = apply_rope(q, start_pos), apply_rope(k, start_pos) # (B,H,T,Dh) -> (B,H,T,Dh)

    # kv cache
    if self.max_context > 0:
      if not hasattr(self, "cache_kv"):
        self.cache_kv = Tensor.zeros(2, self.max_context, B, self.num_heads, self.head_dim, dtype=k.dtype, device=k.device).contiguous().realize()
      self.cache_kv[:, start_pos:start_pos+T, :, :, :].assign(Tensor.stack(k, v)).realize()
      k = self.cache_kv[0, 0:start_pos+T, :, :, :]
      v = self.cache_kv[1, 0:start_pos+T, :, :, :]

    # compute attention
    qk = q.matmul(k.transpose(-1, -2)) / math.sqrt(q.shape[-1]) # (B,H,T,Dh) (B,H,Dh,T) -> (B,H,T,T)
    if is_causal and T > 1:
      mask = Tensor.full((1, 1, T, start_pos+T), float("-inf"), dtype=x.dtype, device=x.device).triu(start_pos+1) # (B,H,T,T)
      qk = qk + mask # (B,H,T,T)
    s = qk.softmax(-1) # (B,H,T,T) -> (B,H,T,T)
    attn = s.matmul(v) # (B,H,T,T) (B,H,T,Dh) -> (B,H,T,Dh)
    attn = attn.transpose(1, 2).reshape(B, T, -1) # (B,H,T,Dh) -> (B,T,D)
    out = self.attn_o(attn) # (B,T,D) -> (B,T,D)
    return out

In [61]:
B, T, D, H = 1, 1, 4, 1
assert D % H == 0, f'hidden D {D=} must be divisible by number of heads {H=}'
x = Tensor.arange(B*T*D).reshape(B,T,D).cast(dtypes.float)

with Stats("MHA Weights:"):
    model = MHA(D, H)
    for p in nn.state.get_parameters(model): p.realize()

with Stats("MHA Forward Pass:"):
    out = model(x).realize()

MHA Weights:        	    22.64 ms,     14940 OPs,      2180 bytes, 659870.91 OPs/sec,  96286.38 bytes/sec,      6.85 OPs/byte
MHA Forward Pass:   	  7520.38 us,       116 OPs,       440 bytes,  15424.76 OPs/sec,  58507.72 bytes/sec,      0.26 OPs/byte


In [53]:
# group-query attention
# https://arxiv.org/abs/2305.13245
class GQA:
  def __init__(self, D:int, H:int, Hkv:int):
    self.num_heads = H # number of heads
    self.num_headskv = Hkv # number of kv heads
    self.head_dim = D // H # head dimension

    self.attn_q = nn.Linear(D, self.head_dim*H, bias=False) # (D,Dh*H)
    self.attn_k = nn.Linear(D, self.head_dim*Hkv, bias=False) # (D,Dh*Hkv)
    self.attn_v = nn.Linear(D, self.head_dim*Hkv, bias=False) # (D,Dh*Hkv)
    self.attn_o = nn.Linear(D, self.head_dim*H, bias=False) # (D,Dh*H)

  def __call__(self, x:Tensor, start_pos:int=0, is_causal=True) -> Tensor:
    # projections
    q = self.attn_q(x) # (B,T,D) -> (B,T,Dh*H)
    k = self.attn_k(x) # (B,T,D) -> (B,T,Dh*Hkv)
    v = self.attn_v(x) # (B,T,D) -> (B,T,Dh*Hkv)

    # reshape
    B, T, _ = x.shape
    q = q.reshape(B, T, self.num_heads, self.head_dim).transpose(1, 2)  # (B,T,D) -> (B,H,T,Dh)
    k = k.reshape(B, T, self.num_headskv, self.head_dim).transpose(1, 2)  # (B,T,Dh*Hkv) -> (B,Hkv,T,Dh)
    v = v.reshape(B, T, self.num_headskv, self.head_dim).transpose(1, 2)  # (B,T,Dh*Hkv) -> (B,Hkv,T,Dh)

    # gqa reshape
    k = k.repeat_interleave(self.num_heads // k.shape[-3], dim=-3) # (B,Hkv,T,Dh) -> (B,H,T,Dh)
    v = v.repeat_interleave(self.num_heads // v.shape[-3], dim=-3) # (B,Hkv,T,Dh) -> (B,H,T,Dh)

    # positional embeddings
    q, k = apply_rope(q, start_pos), apply_rope(k, start_pos) # (B,Hkv,T,Dh) -> (B,Hkv,T,Dh)

    # compute attention
    qk = q.matmul(k.transpose(-1, -2)) / math.sqrt(q.shape[-1]) # (B,H,T,Dh) (B,H,T,Dh) -> (B,H,T,T)
    if is_causal:
      mask = Tensor.full((1, 1, T, start_pos+T), float("-inf"), dtype=x.dtype, device=x.device).triu(start_pos+1) if T > 1 else None # (B,H,T,T)
      qk = qk + mask # (B,H,T,T)
    s = qk.softmax(-1) # (B,H,T,T) -> (B,H,T,T)
    attn = s.matmul(v) # (B,H,T,T) (B,H,T,Dh) -> (B,H,T,Dh)
    attn = attn.transpose(1, 2).reshape(B, T, -1) # (B,H,T,Dh) -> (B,T,D)
    out = self.attn_o(attn) # (B,T,D) -> (B,T,D)
    return out

In [62]:
B, T, D, H, Hkv = 2, 32, 256, 4, 2
assert D % H == 0, f'hidden D {D=} must be divisible by number of heads {H=}'
x = Tensor.arange(B*T*D).reshape(B,T,D).cast(dtypes.float).realize()

with Stats("GQA Weights:"):
    model = GQA(D, H, Hkv)
    for p in nn.state.get_parameters(model): p.realize()

with Stats("GQA Forward Pass:"):
    out = model(x).realize()

GQA Weights:        	   315.18 ms,  43819012 OPs,      6.29 MB,    139.03 MOPs/sec,     19.96 MB/sec,      6.96 OPs/byte
GQA Forward Pass:   	    72.62 ms,  27657024 OPs,      2.25 MB,    380.84 MOPs/sec,     30.99 MB/sec,     12.29 OPs/byte


In [13]:
# multi-query attention
# https://arxiv.org/abs/1911.02150
class MQA(GQA):
  def __init__(self, D:int, H:int):
    # MQA is just like GQA but we set set n_kv_heads to 1
    super().__init__(D, H, 1)

In [14]:
B, T, D, H = 2, 32, 256, 4
assert D % H == 0, f'hidden D {D=} must be divisible by number of heads {H=}'

x = Tensor.arange(B*T*D).reshape(B,T,D).cast(dtypes.float).contiguous()
model = MQA(D, H)
out = model(x)
out.realize()

<Tensor <LB METAL (2, 32, 256) float ShapeTracker(views=(View(shape=(2, 32, 256), strides=(8192, 256, 1), offset=0, mask=None, contiguous=True),))> on METAL with grad None>